In [20]:
from nltkParser import words_to_pos_vector
from mongo_utils import get_mongo_client
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from machine_learning import perform_kmeans, fit_kmeans, perform_hierarchical, run_DBSCAN, Gaussian_Mixture_Models, Isolation_Forest

In [21]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.decomposition import PCA


def plot_clusters(labeled_data, graph_title='Cluster Visualization'):
    # Prepare the data for plotting
    types = [str(key[0]) for key in labeled_data.keys()]
    labels = [data['cluster'] for data in labeled_data.values()]
    vectors = [data['vector'] for data in labeled_data.values()]

    # Converting list of vectors into a 2D array for plotting
    vectors = np.array(vectors)

    # PCA Reduction if necessary
    if vectors.shape[1] > 2:
        pca = PCA(n_components=2)
        vectors = pca.fit_transform(vectors)
        print("Data has been reduced to 2 dimensions using PCA for visualization.")

    # Create unique combinations of type and cluster for markers and colors
    unique_types = sorted(set(types))
    unique_clusters = sorted(set(labels))
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_clusters)))  # Color map for types
    markers = ['o', 's', '^', 'D']  # Extend this list if you have more clusters

    cluster_to_color = {c: col for c, col in zip(unique_clusters, colors)}
    type_to_marker = {t: m for t, m in zip(unique_types, markers)}

    # Create the scatter plot
    plt.figure(figsize=(24, 12))
    for type_val, label, vector in zip(types, labels, vectors):
        color = cluster_to_color[label]
        marker = type_to_marker[type_val]
        plt.scatter(vector[0], vector[1], label=f"Cluster {label} Type {type_val}",
                    color=color, marker=marker, s=100, alpha=0.7)

    plt.title(graph_title, fontsize=16)
    plt.xlabel('Component 1', fontsize=14)
    plt.ylabel('Component 2', fontsize=14)
    plt.grid(True)
    plt.show()


def calculate_cluster_percentages(data):
    cluster_distribution = {}
    for key, value in data.items():
        type_ = key[0]
        cluster = value['cluster']
        if type_ not in cluster_distribution:
            cluster_distribution[type_] = {}
        if cluster not in cluster_distribution[type_]:
            cluster_distribution[type_][cluster] = 0
        cluster_distribution[type_][cluster] += 1

    # Prepare data for DataFrame
    data_for_df = []
    for type_, clusters in cluster_distribution.items():
        for cluster, count in clusters.items():
            data_for_df.append([type_, cluster, count])
            
    # Aggregate totals per type
    type_totals = {}
    for data in data_for_df:
        type_name = data[0]
        count = data[2]
        if type_name in type_totals:
            type_totals[type_name] += count
        else:
            type_totals[type_name] = count

    # Compute totals per cluster per type
    cluster_totals = {}
    for data in data_for_df:
        type_name = data[0]
        cluster_id = data[1]
        count = data[2]
        if type_name not in cluster_totals:
            cluster_totals[type_name] = {}
        if cluster_id not in cluster_totals[type_name]:
            cluster_totals[type_name][cluster_id] = 0
        cluster_totals[type_name][cluster_id] += count

    # Calculate and format the percentage per cluster per type
    output = []
    for type_name, clusters in sorted(cluster_totals.items()):
        output.append(f"Type: {type_name}")
        for cluster_id, count in sorted(clusters.items()):
            percentage = (count / type_totals[type_name]) * 100
            output.append(f"  Cluster {cluster_id}: {percentage:.2f}%")

    return output




# Define color map outside the function
color_map = {
    'gpt-4-turbo-preview': 'yellow',
    'gpt-3.5-turbo': 'red',
    'NYT': 'green',
    'human': 'blue'

}
from sklearn.metrics import precision_score, recall_score, f1_score
def analyze_and_display_clusters(data, num_clusters = 2, title = 'Cluster Distribution'):
    # Aggregate cluster distributions by type
    cluster_distribution = {}
    for key, value in data.items():
        type_ = key[0]
        cluster = value['cluster']
        if type_ not in cluster_distribution:
            cluster_distribution[type_] = {}
        if cluster not in cluster_distribution[type_]:
            cluster_distribution[type_][cluster] = 0
        cluster_distribution[type_][cluster] += 1

    # Prepare data for DataFrame for graph
    data_for_df = []
    for type_, clusters in cluster_distribution.items():
        for cluster, count in clusters.items():
            data_for_df.append([type_, cluster, count])

    # Define AI and human types for the table display
    ai_types = ['gpt-3.5-turbo', 'gpt-4-turbo-preview']
    human_types = ['NYT', 'human']
    cluster_info = {i: {'ai_total': 0, 'human_total': 0, 'total': 0} for i in range(num_clusters)}

    # Process data for AI vs Human content distribution
    for type_, clusters in cluster_distribution.items():
        for cluster, count in clusters.items():
            type_group = 'ai' if type_ in ai_types else 'human'
            cluster_info[cluster][f'{type_group}_total'] += count
            cluster_info[cluster]['total'] += count

    # Determine the majority label in each cluster
    cluster_majority_label = {}
    for cluster_id, info in cluster_info.items():
        if info['ai_total'] >= info['human_total']:
            cluster_majority_label[cluster_id] = 'ai'
        else:
            cluster_majority_label[cluster_id] = 'human'
    
    print("Cluster Majority Label:")
    print(cluster_majority_label)
    
    # Lists to store actual and predicted labels
    y_true = []
    y_pred = []
    
    # Construct labels
    for key, value in data.items():
        actual_type = 'ai' if key[0] in ai_types else 'human'
        assigned_cluster = value['cluster']
        predicted_label = cluster_majority_label[assigned_cluster]
    
        y_true.append(actual_type)
        y_pred.append(predicted_label)
    
    # Compute metrics separately for AI and Human
    labels = ['ai', 'human']
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels)
    accuracy = accuracy_score(y_true, y_pred)
    
    # Display results separately
    print("=== Classification Metrics by Class ===")
    for idx, label in enumerate(labels):
        print(f"\nClass: {label.upper()}")
        print(f"Precision: {precision[idx]:.4f}")
        print(f"Recall:    {recall[idx]:.4f}")
        print(f"F1 Score:  {f1[idx]:.4f}")
        print(f"Support:   {support[idx]} samples")
    
    print("\n=== Overall Accuracy ===")
    print(f"Accuracy: {accuracy:.4f}")


    # DataFrame for table display
    table_data = {
        'Cluster': [],
        'AI Content (%)': [],
        'Human Content (%)': []
    }
    for cluster_id in sorted(cluster_info):
        info = cluster_info[cluster_id]
        ai_percent = (info['ai_total'] / info['total'] * 100) if info['total'] != 0 else 0
        human_percent = (info['human_total'] / info['total'] * 100) if info['total'] != 0 else 0
        table_data['Cluster'].append(cluster_id)
        table_data['AI Content (%)'].append(round(ai_percent, 2))
        table_data['Human Content (%)'].append(round(human_percent, 2))
        
    
    print("\nCluster Distribution:")
    for line in calculate_cluster_percentages(data):
        print(line)

    # Create and display bar graph
    df_graph = pd.DataFrame(data_for_df, columns=['Type', 'Cluster', 'Count'])
    plt.figure(figsize=(20, 6))
    sns.barplot(x='Cluster', y='Count', hue='Type', data=df_graph, palette=color_map)
    plt.title(title, fontsize=16)
    plt.xlabel('Cluster')
    plt.ylabel('Count')
    plt.legend(title='Type')
    plt.show()

    # Create and display table as an image
    df_table = pd.DataFrame(table_data)
    fig, ax = plt.subplots(figsize=(8, num_clusters * 0.5))  # Adjust size appropriately
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=df_table.values, colLabels=df_table.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)  # Adjust table scale
    plt.show()
    
    return cluster_majority_label


In [22]:
def fetch_embedded_vector(collection):
    # Fetch only the necessary fields
    documents = collection.find({}, {'_id': 1, 'Model Used': 1, 'text-embedding-3-small': 1})
    results = {}
        
    for doc in documents:

        if collection.name == 'NYTimeArticle':
            type_id = ('NYT', str(doc['_id']))
        else:
            # Using 'Model Used' or 'human' if it doesn't exist
            type_id = (doc.get('Model Used', 'human'), str(doc['_id']))
        # Directly assign the array of floats to the tuple key
        results[type_id] = doc.get('text-embedding-3-small', [])
    return results

def fetch_pos_vector(collection):
    # Fetch only the necessary fields
    documents = collection.find({}, {'_id': 1, 'Model Used': 1, 'wordFamilyContent': 1})
    results = {}
    
    for doc in documents:
        if collection.name == 'NYTimeArticle':
            type_id = ('NYT', str(doc['_id']))
        else:
            # Using 'Model Used' or 'human' if it doesn't exist
            type_id = (doc.get('Model Used', 'human'), str(doc['_id']))
        # Directly assign the array of floats to the tuple key
        results[type_id] = words_to_pos_vector(doc.get('wordFamilyContent', []))
    return results

In [23]:
# Initialize the MongoDB Atlas Connection
username = 'lilousicardnoel'
cluster_url = 'cluster0.figrf53.mongodb.net'
db_name = 'newsArticle'

mongo_client = get_mongo_client(username, cluster_url, db_name)
db = mongo_client[db_name]
topicsForAI = db['topicsForAI']
chatGPT_articles = db['ChatGPT-Articles']
pro_quest_articles = db['ProQuest-Articles']
nyTimes_articles = db['NYTimeArticle']
# print the length of the collections
print(f"ChatGPT-Articles: {chatGPT_articles.count_documents({})}")
print(f"ProQuest-Articles: {pro_quest_articles.count_documents({})}")
print(f"NYTimeArticle: {nyTimes_articles.count_documents({})}")

ChatGPT-Articles: 398
ProQuest-Articles: 1900
NYTimeArticle: 191


In [24]:
# Fetch the embedded vectors for each article
chatGPT_articles_data = fetch_embedded_vector(chatGPT_articles)
pro_quest_articles_data = fetch_embedded_vector(pro_quest_articles)
nyTimes_articles_data = fetch_embedded_vector(nyTimes_articles)
# getting pos vector
chatGPT_articles_pos = fetch_pos_vector(chatGPT_articles)
pro_quest_articles_pos = fetch_pos_vector(pro_quest_articles)
nyTimes_articles_pos = fetch_pos_vector(nyTimes_articles)

KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import train_test_split

def split_dictionary(data_dict, test_size=0.2):
    """
    Splits a dictionary into two smaller dictionaries (train and test) while maintaining key-value pairs.
    
    Parameters:
        data_dict (dict): The dictionary to split.
        test_size (float): The proportion of the dictionary to include in the test split.
    
    Returns:
        tuple: (train_dict, test_dict) - Two dictionaries split according to test_size.
    """
    keys = list(data_dict.keys())  # Extract keys
    train_keys, test_keys = train_test_split(keys, test_size=test_size)  # Split keys

    train_dict = {key: data_dict[key] for key in train_keys}  # Create training dictionary
    test_dict = {key: data_dict[key] for key in test_keys}  # Create testing dictionary

    return train_dict, test_dict

def shuffle_dictionary(data_dict):
    """
    Shuffles the key-value pairs of a dictionary.
    
    Parameters:
        data_dict (dict): The dictionary to shuffle.
    
    Returns:
        dict: The shuffled dictionary.
    """
    keys = list(data_dict.keys())
    np.random.shuffle(keys)
    return {key: data_dict[key] for key in keys}

In [ ]:
# getting embedded vector
pro_quest_training_emb, pro_quest_testing_emb = split_dictionary(pro_quest_articles_data, test_size=0.2)
chatGPT_training_emb, chatGPT_testing_emb = split_dictionary(chatGPT_articles_data, test_size=0.2)
nyTimes_training_emb, nyTimes_testing_emb = split_dictionary(nyTimes_articles_data, test_size=0.2)
# getting pos vector
pro_quest_training_pos, pro_quest_testing_pos = split_dictionary(pro_quest_articles_pos, test_size=0.2)
chatGPT_training_pos, chatGPT_testing_pos = split_dictionary(chatGPT_articles_pos, test_size=0.2)
nyTimes_training_pos, nyTimes_testing_pos = split_dictionary(nyTimes_articles_pos, test_size=0.2)

# Display the length of the training and testing data
print(f"ProQuest Training: {len(pro_quest_training_emb)}")
print(f"ProQuest Testing: {len(pro_quest_testing_emb)}")
print(f"ChatGPT Training: {len(chatGPT_training_emb)}")
print(f"ChatGPT Testing: {len(chatGPT_testing_emb)}")
print(f"NYTimes Training: {len(nyTimes_training_emb)}")
print(f"NYTimes Testing: {len(nyTimes_testing_emb)}")

# Combine the training data 
training_data_emb = {**pro_quest_training_emb, **chatGPT_training_emb, **nyTimes_training_emb}
training_data_emb = shuffle_dictionary(training_data_emb)
training_data_pos = {**pro_quest_training_pos, **chatGPT_training_pos, **nyTimes_training_pos}
training_data_pos = shuffle_dictionary(training_data_pos)
# Combine the testing data
testing_data_emb = {**pro_quest_testing_emb, **chatGPT_testing_emb, **nyTimes_testing_emb}
testing_data_emb = shuffle_dictionary(testing_data_emb)
testing_data_pos = {**pro_quest_testing_pos, **chatGPT_testing_pos, **nyTimes_testing_pos}
testing_data_pos = shuffle_dictionary(testing_data_pos)

# Display the length of the combined training and testing data
print(f"Training: {len(training_data_emb)}")
print(f"Testing: {len(testing_data_emb)}")

In [ ]:
# Print the first key-value pair in the training data
key, value = next(iter(training_data_emb.items()))
print(key)

# **KMeans Clustering with embedded vectors**

In [ ]:
kmeans_emb, kmeans_results_emb = perform_kmeans(training_data_emb, 5, 13)
kmeans_emb_distribution = calculate_cluster_percentages(kmeans_results_emb)
cluster_majority_label_emb = analyze_and_display_clusters(kmeans_results_emb, kmeans_emb.n_clusters, 'KMeans Cluster Distribution')

In [ ]:
# test the model
labeled_data_emb = fit_kmeans(kmeans_emb, testing_data_emb)

In [ ]:
# Plot the clusters
plot_clusters(labeled_data_emb, 'KMeans Cluster EMB Visualization')
analyze_and_display_clusters(labeled_data_emb, kmeans_emb.n_clusters, 'KMeans Cluster EMB Distribution')

# **KMeans Clustering with POS vectors**

In [ ]:
seed = 774
num_clusters = 9
print(f"Number of Clusters: {num_clusters}, Seed: {seed}")
kmeans_pos, kmeans_results_pos = perform_kmeans(training_data_pos, num_clusters, seed)
kmeans_distribution = calculate_cluster_percentages(kmeans_results_pos)
cluster_majority_label_pos = analyze_and_display_clusters(kmeans_results_pos, kmeans_pos.n_clusters, 'KMeans Cluster POS Distribution')

In [ ]:
labeled_data_pos = fit_kmeans(kmeans_pos, testing_data_pos)

In [ ]:
plot_clusters(labeled_data_pos, 'KMeans Cluster POS Visualization')
analyze_and_display_clusters(labeled_data_pos, kmeans_pos.n_clusters, 'KMeans Cluster POS Distribution')

# Hierarchical Clustering with embedded vectors

In [ ]:
"1 distance threshold = 0.9538 accuracy, 851 cluster"
"1.25 distance threshold = 0.9392 accuracy, 495 cluster"
"1.5 distance threshold = 0.9302 accuracy, 291 cluster"
"2 distance threshold = 0.9090 accuracy, 127 cluster"
hierarchical_results_emb, hierarchical_centroids_emb = perform_hierarchical(training_data_emb, num_clusters=None, distance_threshold=1.25)
hierarchical_emb_distribution = calculate_cluster_percentages(hierarchical_results_emb)
cluster_majority_label_hierarchical_emb = analyze_and_display_clusters(hierarchical_results_emb, len(hierarchical_centroids_emb), 'Hierarchical Cluster EMB Distribution')


In [ ]:
from src.Machine_Learning.machine_learning import test_hierarchical

hierarchical_test_results_emb = test_hierarchical(testing_data_emb, hierarchical_centroids_emb)
hierarchical_test_emb_distribution = calculate_cluster_percentages(hierarchical_test_results_emb)
analyze_and_display_clusters(hierarchical_test_results_emb, len(hierarchical_centroids_emb), 'Hierarchical Cluster EMB Distribution')


# **Hierarchical Clustering with POS vectors**

# Gaussian Mixture Models Clustering with embedded vectors

# **Gaussian Mixture Models Clustering with POS vectors**

# Retest DBSCAN and Isolation Forest